In [1]:
import numpy as np
import pandas as pd
import glob

# Win Files

In [2]:
files = glob.glob("../data/spcsv/dwbfpricesukwin*.csv")

dfs = []
for f in files:
    df = pd.read_csv(f)

    df.columns = (df.columns.str.strip().str.lower())
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df = df.dropna()

# Normalise data

## Assign location

In [3]:
df["location"] = (df.menu_hint
    .str.extract(r"(?:GB / )?([A-Za-z ]+)")[0]
    .str.strip())
lmap = {
    "Aint": "Aintree",
    "Bang": "Bangor On Dee",
    "Bangor": "Bangor On Dee",
    "Bangor on Dee": "Bangor On Dee",
    "Bev": "Beverly",
    "Brig": "Brighton",
    "Carl": "Carlisle",
    "Cart": "Cartmel",
    "Catt": "Cattertick",
    "ChelmC": "Chelmsford City",
    "ChelmC PA": "Chelmsford City",
    "Chelt": "Cheltenham",
    "Chep": "Chepstow",
    "Chest": "Chester",
    "Donc": "Doncaster",
    "Epsm": "Epsom",
    "Extr": "Exeter",
    "Fake": "Fakenham",
    "FfosL": "Ffoss Las",
    "Font": "Fontwell",
    "GLghs": "Chelmsford City",
    "Good": "Goodwood",
    "Ham": "Hamilton",
    "Hayd": "Haydock",
    "Here": "Hereford",
    "Hex": "Hexham",
    "Hunt": "Huntingdon",
    "Kemp": "Kempton",
    "Leic": "Leicester",
    "Ling": "Lingfield",
    "Lud": "Ludlow",
    "MrktR": "Market Rasen",
    "Muss": "Musselburgh",
    "Newb": "Newbury",
    "Newc": "Newcastle",
    "Newm": "Newmarket",
    "Newt": "Newton Abbot",
    "Nott": "Nottingham",
    "Plump": "Plumpton",
    "Ponte": "Pontefract",
    "Redc": "Redcar",
    "Salis": "Salisbury",
    "Sand": "Sandown",
    "Sedge": "Sedgefield",
    "Sthl": "Southwell",
    "Strat": "Stratford",
    "Taun": "Taunton",
    "Uttox": "Uttoxeter",
    "Warw": "Warwick",
    "Weth": "Wetherby",
    "Winc": "Wincanton",
    "Wind": "Windsor",
    "Wolv": "Wolverhampton",
    "Worc": "Worcester",
    "Yarm": "Yarmouth"
}
df.location = df.location.replace(lmap)

## Fix poorly formatted runner names

In [4]:
df.selection_name = df.selection_name.str.replace(r"^\d+\.? ", "", regex=True)

## Distance

In [5]:
df = df[df.event_name.str.contains(r"^\d")] # remove poor formatting
df["distance_f"] = (df.event_name.str.strip().str.extract(r"(?:(\d+)m)?(?:(\d)f?)?").fillna(0).astype(int) * [8, 1]).sum(axis=1)
df.loc[(df.distance_f == 0) | (df.distance_f >= 40), "distance_f"] = np.nan

## Race type

In [6]:
df["race_type"] = df.event_name.str.strip().str.split().str[1:].str.join(" ")

## Date

In [7]:
df["race_time"] = pd.to_datetime(df.event_dt, dayfirst=True)
df["event_date"] = df.race_time.dt.date

## Assign event IDs

In [8]:
df["race_id"] = df["event_id"]
df["event_id"] = df.groupby(["location", "event_date"]).ngroup()

## Implied probability

In [9]:
df["implied_odds"] = 1/df.bsp

## Remove redundancies

In [10]:
df = df.drop(columns=["menu_hint", "event_name", "event_dt"])
df = df.dropna()

## Remove Duplicates

In [11]:
df = df[df.race_type!="TBP"]
df = df.drop_duplicates(["race_time", "selection_id"])

## Assign types

In [12]:
df = df.astype({
    "event_id": "int64",
    "selection_id": "int64",
    "selection_name": "string",
    "win_lose": "bool",
    "bsp": "float64",
    "ppwap": "float32",
    "morningwap": "float32",
    "ppmax": "float32",
    "ppmin": "float32",
    "ipmax": "float32",
    "ipmin": "float32",
    "morningtradedvol": "float32",
    "pptradedvol": "float32",
    "iptradedvol": "float32",
    "location": "string",
    "distance_f": "int8",
    "race_type": "string",
    "race_time": "datetime64[s]",
    "event_date": "datetime64[s]",
    "race_id": "int64",
    "implied_odds": "float64"
})

In [13]:
win=df

# Place Files

In [14]:
files = glob.glob("../data/spcsv/dwbfpricesukplace*.csv")

dfs = []
for f in files:
    df = pd.read_csv(f)

    df.columns = (df.columns.str.strip().str.lower())
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df = df.dropna()

In [15]:
place = df

## Get Relative Info

In [16]:
place["race_time"] = pd.to_datetime(place.event_dt, dayfirst=True)
place["event_date"] = place.race_time.dt.date
place["race_id"] = place["event_id"]

place["place_bsp"] = place.bsp
place["place_win_lose"] = place.win_lose
place["place_n"] = place.groupby("race_id").win_lose.transform(lambda s: s.sum())

place["race_type"] = place.event_name.str.strip().str.split().str[1:].str.join(" ")

place = place[place.race_type!="TBP"]
place = place.drop_duplicates(["race_time", "selection_id"])

place = place[["race_time", "selection_id", "place_bsp", "place_n", "place_win_lose"]]
place = place.astype({
    "selection_id": "int64",
    "place_bsp": "float64",
    "place_win_lose": "bool",
    "place_n": "int8",
    "race_time": "datetime64[s]",
})

place = place.set_index(["race_time", "selection_id"])

In [17]:
df = win.join(place, on=["race_time", "selection_id"], how="inner", validate="one_to_one")

In [18]:
win[win.selection_id==23320215].sort_values("race_time")

,event_id,selection_id,selection_name,win_lose,bsp,ppwap,morningwap,ppmax,ppmin,ipmax,...,morningtradedvol,pptradedvol,iptradedvol,location,distance_f,race_type,race_time,event_date,race_id,implied_odds
426181,24371,23320215,Wayfinder,True,8.200000,7.904810,7.072127,11.500000,6.40,13.50,...,4323.060059,29524.279297,62530.781250,Worcester,16,Mdn NHF,2019-10-10 15:20:00,2019-10-10,163468620,0.121951
662524,20071,23320215,Wayfinder,False,2.500000,2.631745,3.833425,4.300000,2.40,1000.00,...,3232.520020,188673.062500,48999.281250,Stratford,22,Mdn Hrd,2019-10-31 13:45:00,2019-10-31,164418941,0.400000
617325,6549,23320215,Wayfinder,False,7.087269,9.187168,7.888544,12.000000,6.80,1000.00,...,2205.000000,49991.898438,13474.700195,Ffoss Las,20,Mdn Hrd,2019-11-28 12:25:00,2019-11-28,165631178,0.141098
797035,21429,23320215,Wayfinder,False,13.226878,11.748010,13.020338,14.500000,9.20,1000.00,...,558.159973,22715.060547,2596.590088,Warwick,26,Hcap Hrd,2020-02-08 15:50:00,2020-02-08,168474018,0.075604
995007,8295,23320215,Wayfinder,True,2.732028,2.890994,3.685078,4.300000,2.62,27.00,...,4688.060059,176689.000000,144508.015625,Hereford,25,Hcap Hrd,2020-03-16 15:50:00,2020-03-16,170166856,0.366029
429645,6553,23320215,Wayfinder,False,5.700000,5.270741,7.722464,7.600000,5.40,1000.00,...,444.559998,34775.460938,16921.599609,Ffoss Las,24,Nov Hcap Chs,2020-10-01 14:43:00,2020-10-01,173674290,0.175439
323349,4525,23320215,Wayfinder,True,2.248196,2.398457,2.882199,3.650000,2.22,4.00,...,11083.580078,243549.781250,185632.625000,Chepstow,26,Nov Hcap Chs,2020-10-27 13:50:00,2020-10-27,174684459,0.444801
1461274,1233,23320215,Wayfinder,False,12.500000,9.225430,14.709472,20.000000,7.80,1000.00,...,475.260010,31466.640625,2458.070068,Bangor On Dee,24,Hcap Chs,2021-12-10 13:30:00,2021-12-10,192176181,0.080000
932927,13977,23320215,Wayfinder,False,18.555000,11.773979,10.041694,20.000000,8.63,1000.00,...,4802.240234,41247.281250,8130.370117,Newbury,26,Hcap Chs,2021-12-29 14:35:00,2021-12-29,192808886,0.053894
582853,4559,23320215,Wayfinder,False,6.150146,5.527942,6.218313,11.000000,5.50,1000.00,...,4085.860107,47315.800781,32649.000000,Chepstow,26,Hcap Chs,2022-02-04 15:30:00,2022-02-04,194193558,0.162598


# Observe result

In [19]:
df

,event_id,selection_id,selection_name,win_lose,bsp,ppwap,morningwap,ppmax,ppmin,ipmax,...,location,distance_f,race_type,race_time,event_date,race_id,implied_odds,place_bsp,place_n,place_win_lose
0,1113,3997328,Entertain Me,False,3.178476,2.896974,2.665789,3.50,2.57,1000.00,...,Bangor On Dee,20,Hcap Chs,2012-12-14 12:30:00,2012-12-14,107735200,0.314616,1.747948,2,False
1,1113,4209712,Miss Kalifa,False,7.018690,6.103194,6.263207,7.20,4.90,1000.00,...,Bangor On Dee,20,Hcap Chs,2012-12-14 12:30:00,2012-12-14,107735200,0.142477,2.676774,2,False
2,1113,4334735,Evella,False,8.990408,8.730378,8.091962,10.00,7.40,1000.00,...,Bangor On Dee,20,Hcap Chs,2012-12-14 12:30:00,2012-12-14,107735200,0.111230,3.043535,2,True
3,1113,4625415,Midnight Macarena,True,2.400000,2.662097,2.940964,2.98,2.38,3.35,...,Bangor On Dee,20,Hcap Chs,2012-12-14 12:30:00,2012-12-14,107735200,0.416667,1.490000,2,True
4,11440,6283559,Myzamour,False,28.720559,23.676031,22.152115,32.00,14.50,1000.00,...,Lingfield,8,Claim Stks,2012-12-14 12:10:00,2012-12-14,107730583,0.034818,7.451014,2,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1633801,10428,56816865,Al Khawaneej River,False,32.104346,26.470301,19.911900,50.00,16.50,1000.00,...,Kempton,8,Hcap,2024-03-02 17:05:00,2024-03-02,225506599,0.031148,8.200000,3,False
1633802,10428,5473514,Classic Speed,True,5.109491,4.993000,4.637700,6.57,4.28,5.30,...,Kempton,8,Hcap,2024-03-02 16:30:00,2024-03-02,225506578,0.195714,2.086160,3,True
1633803,10428,47883996,Natacata,False,2.631536,2.839900,3.697500,4.03,2.56,1000.00,...,Kempton,16,Hcap,2024-03-02 19:15:00,2024-03-02,225506640,0.380006,1.318888,3,True
1633804,23971,57238190,Ventura Empress,False,54.101238,30.646900,15.417100,65.00,17.50,780.00,...,Wolverhampton,7,Hcap,2024-03-02 20:30:00,2024-03-02,225506716,0.018484,12.500000,3,False


## Events

In [20]:
events = df[["event_id", "location", "event_date"]].drop_duplicates().set_index("event_id")
events

,location,event_date
event_id,,
1113,Bangor On Dee,2012-12-14
11440,Lingfield,2012-12-14
3988,Cheltenham,2012-12-14
22976,Wolverhampton,2012-12-14
8099,Haydock,2022-11-30
...,...,...
14039,Newbury,2024-03-02
10428,Kempton,2024-03-02
5454,Doncaster,2024-03-02


## Races

In [21]:
races = df[["race_id", "race_type", "race_time", "distance_f", "event_id", "place_n"]].drop_duplicates().set_index("race_id")
races["race_no"] = races.sort_values(["event_id","race_time"]).groupby("event_id").cumcount() + 1
races

,race_type,race_time,distance_f,event_id,place_n,race_no
race_id,,,,,,
107735200,Hcap Chs,2012-12-14 12:30:00,20,1113,2,2
107730583,Claim Stks,2012-12-14 12:10:00,8,11440,2,1
107730585,Mdn Stks,2012-12-14 12:40:00,7,11440,3,2
107730587,Nursery,2012-12-14 13:10:00,7,11440,3,3
107730589,Hcap,2012-12-14 13:45:00,16,11440,3,4
...,...,...,...,...,...,...
225506541,Nov Hcap Chs,2024-03-02 16:00:00,21,9118,2,5
225506552,Hrd,2024-03-02 16:35:00,16,9118,2,6
225506751,Nov Hcap Hrd,2024-03-02 14:25:00,16,14039,3,3


## Runners

In [22]:
runners = df[[
    "selection_id",
    "race_id",
    "selection_name",
    "win_lose",
    "place_win_lose",
    "bsp",
    "place_bsp",
    "implied_odds",
    "ppwap",
    "morningwap",
    "ppmax",
    "ppmin",
    "ipmax",
    "ipmin",
    "morningtradedvol",
    "pptradedvol",
    "iptradedvol"
]].set_index("selection_id")
runners

,race_id,selection_name,win_lose,place_win_lose,bsp,place_bsp,implied_odds,ppwap,morningwap,ppmax,ppmin,ipmax,ipmin,morningtradedvol,pptradedvol,iptradedvol
selection_id,,,,,,,,,,,,,,,,
3997328,107735200,Entertain Me,False,False,3.178476,1.747948,0.314616,2.896974,2.665789,3.50,2.57,1000.00,2.82,2966.399902,232305.703125,38222.820312
4209712,107735200,Miss Kalifa,False,False,7.018690,2.676774,0.142477,6.103194,6.263207,7.20,4.90,1000.00,7.00,1175.819946,103042.296875,15779.820312
4334735,107735200,Evella,False,True,8.990408,3.043535,0.111230,8.730378,8.091962,10.00,7.40,1000.00,3.15,425.380005,44032.660156,42255.878906
4625415,107735200,Midnight Macarena,True,True,2.400000,1.490000,0.416667,2.662097,2.940964,2.98,2.38,3.35,1.01,2186.739990,271099.906250,260028.406250
6283559,107730583,Myzamour,False,True,28.720559,7.451014,0.034818,23.676031,22.152115,32.00,14.50,1000.00,11.00,266.940002,15123.620117,2052.979980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56816865,225506599,Al Khawaneej River,False,False,32.104346,8.200000,0.031148,26.470301,19.911900,50.00,16.50,1000.00,27.00,145.279999,3890.159912,920.619995
5473514,225506578,Classic Speed,True,True,5.109491,2.086160,0.195714,4.993000,4.637700,6.57,4.28,5.30,1.01,1548.060059,29416.730469,19033.529297
47883996,225506640,Natacata,False,True,2.631536,1.318888,0.380006,2.839900,3.697500,4.03,2.56,1000.00,2.22,2172.250000,47571.628906,21038.949219


# Store

In [23]:
events.to_parquet("../data/events.parquet")
races.to_parquet("../data/races.parquet")
runners.to_parquet("../data/runners.parquet")